# 🌳 Bayes Tree — Tutorial

This notebook walks through using the Bayes Tree engine to structure evidence,
run Monte Carlo simulations, and interpret results.

## Installation

```bash
pip install bayes-tree
```

Or clone the repo and install from source.

In [ ]:
import sys
sys.path.insert(0, '..')  # if running from examples/ or tutorial/ directory

from bayes_tree import (
    run_simulation, bayes_upd, to_lo, from_lo,
    sim_root, sts, validate_node, __version__
)
import yaml

print(f"Bayes Tree v{__version__}")

## 1. Building Your First Evidence Tree

Every analysis starts with a **yes/no hypothesis** and a **prior probability**.
Then you add evidence branches, each with a **likelihood ratio (LR) interval**.

In [ ]:
# Define an evidence tree as a Python dict (same structure as YAML)
my_tree = {
    "node": "Should I accept this job offer?",
    "prior": 0.50,
    "children": [
        {
            "node": "50% salary increase",
            "lr_min": 2.0,
            "lr_max": 5.0,
            "evidence_type": "for",
        },
        {
            "node": "Longer commute (1.5 hours)",
            "lr_min": 0.3,
            "lr_max": 0.7,
            "evidence_type": "against",
        },
        {
            "node": "Strong company growth trajectory",
            "lr_min": 1.5,
            "lr_max": 3.0,
            "evidence_type": "for",
        },
        {
            "node": "Would lose current team I enjoy",
            "lr_min": 0.4,
            "lr_max": 0.8,
            "evidence_type": "against",
        },
    ],
}

print(f"Hypothesis: {my_tree['node']}")
print(f"Prior: {my_tree['prior']:.0%}")
print(f"Evidence branches: {len(my_tree['children'])}")

## 2. Understanding Likelihood Ratios

| LR | Meaning | Example |
|----|---------|---------|
| 10+ | Strong support | DNA match at crime scene |
| 2–5 | Moderate support | Witness places suspect nearby |
| 1 | Neutral — no update | Irrelevant information |
| 0.2–0.5 | Moderate counter-evidence | Alibi from one friend |
| < 0.1 | Strong counter-evidence | Verified alibi with video |

The key insight: **LR > 1** supports the hypothesis, **LR < 1** opposes it.

In [ ]:
# Single Bayesian update: prior + evidence → posterior
prior = 0.5

print("Single evidence updates from prior = 50%:")
print(f"  LR = 0.1 (strong against): {bayes_upd(prior, 0.1):.1%}")
print(f"  LR = 0.5 (moderate against): {bayes_upd(prior, 0.5):.1%}")
print(f"  LR = 1.0 (neutral):         {bayes_upd(prior, 1.0):.1%}")
print(f"  LR = 2.0 (moderate for):     {bayes_upd(prior, 2.0):.1%}")
print(f"  LR = 10  (strong for):       {bayes_upd(prior, 10.0):.1%}")

## 3. Running the Simulation

The engine runs Monte Carlo simulations, sampling each LR from its interval,
and combines all branches via log-odds summation.

In [ ]:
# Run 10,000 simulations
results = run_simulation(my_tree, n_sim=10_000)

s = results['stats']
print(f"Posterior median: {s['median']:.2%}")
print(f"Posterior mean:   {s['mean']:.2%}")
print(f"90% CI:           [{s['p5']:.2%} – {s['p95']:.2%}]")
print(f"Std:              {s['std']:.2%}")
print(f"\nEffective LR (median): {results['lr_stats']['median']:.3f}")

## 4. Visualizing Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax1.hist(results['posteriors'], bins=30, color='#3b82f6',
         edgecolor='#1e40af', alpha=0.85)
ax1.axvline(s['median'], color='red', linestyle='--', label=f'Median: {s["median"]:.1%}')
ax1.set_xlabel('Posterior probability')
ax1.set_ylabel('Frequency')
ax1.set_title('Posterior Distribution')
ax1.legend()

# Importance
imp = results['importance']
names = [i['name'][:30] for i in imp]
deltas = [i['delta'] for i in imp]
colors = ['#22c55e' if d > 0 else '#ef4444' for d in deltas]
ax2.barh(names, [abs(d) for d in deltas], color=colors)
ax2.set_xlabel('|Δ posterior|')
ax2.set_title('Importance Ranking')

plt.tight_layout()
plt.show()

## 5. Sensitivity Analysis

How sensitive is the conclusion to the choice of prior?

In [ ]:
# Prior sensitivity sweep
priors = np.linspace(0.05, 0.95, 19)
medians = []
ci_low = []
ci_high = []

for p in priors:
    sweep_data = {**my_tree, "prior": float(p)}
    posteriors = [sim_root(sweep_data)[0] for _ in range(3000)]
    ss = sts(posteriors)
    medians.append(ss['median'])
    ci_low.append(ss['p5'])
    ci_high.append(ss['p95'])

plt.figure(figsize=(8, 5))
plt.fill_between(priors, ci_low, ci_high, alpha=0.2, color='#3b82f6', label='90% CI')
plt.plot(priors, medians, 'b-', linewidth=2, label='Median posterior')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='No evidence (prior = posterior)')
plt.xlabel('Prior probability')
plt.ylabel('Posterior probability')
plt.title('Prior Sensitivity: How much does your starting point matter?')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Loading from YAML Files

Real analyses are typically stored as YAML files:

In [ ]:
# Load an example YAML
with open('../examples/napoleon.yaml', encoding='utf-8') as f:
    napoleon = yaml.safe_load(f)

print(f"Hypothesis: {napoleon['node']}")
print(f"Prior: {napoleon['prior']}")
print(f"Evidence branches: {len(napoleon['children'])}")
print()

# Run simulation
results = run_simulation(napoleon, n_sim=10_000)
s = results['stats']
print(f"Posterior: {s['median']:.2%} (90% CI: [{s['p5']:.2%} – {s['p95']:.2%}])")
print(f"\nVerdict: {'Likely YES' if s['median'] > 0.5 else 'Likely NO'} "
      f"(confidence: {abs(s['median'] - 0.5) * 2:.0%})")

## 7. JSON Output for Pipelines

The CLI supports `--format json` for programmatic use:

In [ ]:
import json
from bayes_tree.cli import _format_json

# Get structured JSON output
json_output = json.loads(_format_json(results, napoleon))

print(f"Hypothesis: {json_output['hypothesis']}")
print(f"Posterior median: {json_output['posterior']['median']:.4f}")
print(f"Effective LR: {json_output['effective_lr']['median']:.4f}")
print(f"\nTop 3 most important branches:")
for i, imp in enumerate(json_output['importance'][:3], 1):
    print(f"  {i}. {imp['name']} (Δ = {imp['delta']:+.4f})")

## 8. Validation

The engine validates your tree for logical consistency:

In [ ]:
# This tree has a logical error: evidence_type='for' but LR < 1
bad_tree = {
    "node": "Test",
    "prior": 0.5,
    "children": [
        {
            "node": "Claims to support but actually opposes",
            "lr_min": 0.1,
            "lr_max": 0.3,
            "evidence_type": "for",  # ← conflict!
        }
    ],
}

warnings = validate_node(bad_tree)
for w in warnings:
    print(w)

## Next Steps

- **CLI**: `bayes-tree my_analysis.yaml --format json`
- **GUI**: `python bayes_tree_gui.py` for interactive editing
- **PDF reports**: Export professional reports via the GUI
- **More examples**: See the `examples/` directory

---

*© 2026 Ari-Pekka Sihvonen · MIT License · [GitHub](https://github.com/sihvoar/belief-engine)*